In [ ]:
# Classification 
# Logistic Regression Intuition. Sigmoid & Softmax.
# Decision Tree, SVM
# Classification script with ROC/AUC
# Propability that a input belongs to a class.
# Output as 0 or 1.
# Threshold/Proability >= 0.5 or custom. Decides class 0 or 1.
# 𝑃(𝑦=1∣𝑥)=1/1+e-(B0+B1x).
# Loss Function. Log Loss(Cross-Entropy instead of MSE).
# L= -1/N ∑[ylog(Ŷ)+(1-y)log(1-Ŷ)] (Minimize log loss)
# Binary: Survived categorized into ages (0/1)
# Multiclass: AgeGroup (Child/Adult/Elderly)

import pandas as pd

# preventing truncated output
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, roc_curve, roc_auc_score

# Loading dataset
data = pd.read_csv("The_Titanic.csv")

# Preprocess: converting 'Sex' to numbers (male=0, female=1)
data['Sex'] = data['Sex'].map({'male':0, 'female':1})

# Age groups column creation
def assign_age_group(age):
    if pd.isnull(age):
        return "Unknown"
    elif age < 18:
        return "Child"
    elif age >= 60:
        return "Elderly"
    else:
        return "Adult"

data['AgeGroup'] = data['Age'].apply(assign_age_group)

# Drop missing values (keeps only complete rows)
data = data.dropna(subset=['Age','Fare','Sex','Survived'])

# Binary Classification. Target = Survived
# Selected features (Age, Fare, Sex) and target is (Survived)
X_bin = data[['Age','Fare','Sex']]
y_bin = data['Survived']

# Spliting into training (80%) and testing (20%) with stratifing.
X_train_bin, X_test_bin, y_train_bin, y_test_bin = train_test_split(
    X_bin, y_bin, test_size=0.2, random_state=42, stratify=y_bin
)

# The models I chose to train
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Decision Tree": DecisionTreeClassifier(max_depth=4, random_state=42),
    "SVM": SVC(kernel='rbf', probability=True, random_state=42)
}

results_bin = {}

for name, model in models.items():
    model.fit(X_train_bin, y_train_bin)
    y_pred_bin = model.predict(X_test_bin)
    
    acc = accuracy_score(y_test_bin, y_pred_bin)
    cm = confusion_matrix(y_test_bin, y_pred_bin)
    report = classification_report(y_test_bin, y_pred_bin, target_names=["Not Survived","Survived"])
    
    # ROC/AUC
    y_prob = model.predict_proba(X_test_bin)[:,1]  # probability of class 1
    fpr, tpr, thresholds = roc_curve(y_test_bin, y_prob)
    auc = roc_auc_score(y_test_bin, y_prob)
    
    results_bin[name] = {"accuracy": acc, "confusion_matrix": cm, "report": report, "fpr": fpr, "tpr": tpr, "auc": auc}
    
    print(f"\n{name} (Binary: Survived)")
    print("Accuracy:", acc)
    print("Confusion Matrix:\n", cm)
    print("Classification Report:\n", report)
    print("AUC:", auc)

# Multiclass Classification. Target = AgeGroup
X_multi = data[['Fare','Sex']]
y_multi = data['AgeGroup']

X_train_multi, X_test_multi, y_train_multi, y_test_multi = train_test_split(
    X_multi, y_multi, test_size=0.2, random_state=42, stratify=y_multi
)

results_multi = {}

for name, model in models.items():
    model.fit(X_train_multi, y_train_multi)
    y_pred_multi = model.predict(X_test_multi)
    
    acc = accuracy_score(y_test_multi, y_pred_multi)
    cm = confusion_matrix(y_test_multi, y_pred_multi, labels=model.classes_)
    report = classification_report(y_test_multi, y_pred_multi)
    
    results_multi[name] = {"accuracy": acc, "confusion_matrix": cm, "report": report}
    
    print(f"\n=== {name} (Multiclass: AgeGroup) ===")
    print("Accuracy:", acc)
    print("Confusion Matrix:\n", cm)
    print("Classification Report:\n", report)

# Saving the report
with open("titanic_results.txt", "w") as f:
    for name, res in results_bin.items():
        f.write(f"\n{name} (Binary: Survived)\n")
        f.write("Accuracy: " + str(res["accuracy"]) + "\n")
        f.write("Confusion Matrix:\n" + str(res["confusion_matrix"]) + "\n")
        f.write("Classification Report:\n" + res["report"] + "\n")
        f.write("AUC: " + str(res["auc"]) + "\n")
    for name, res in results_multi.items():
        f.write(f"\n{name} (Multiclass: AgeGroup)\n")
        f.write("Accuracy: " + str(res["accuracy"]) + "\n")
        f.write("Confusion Matrix:\n" + str(res["confusion_matrix"]) + "\n")
        f.write("Classification Report:\n" + res["report"] + "\n")

print("\nAll results saved to titanic_results.txt")

# Visualization 1: Accuracy comparison
# Accuracy comparison
plt.figure(figsize=(8,5))
sns.barplot(x=list(results_bin.keys()), y=[results_bin[m]["accuracy"] for m in results_bin], color="skyblue", label="Binary (Survived)")
sns.barplot(x=list(results_multi.keys()), y=[results_multi[m]["accuracy"] for m in results_multi], color="lightgreen", alpha=0.6, label="Multiclass (AgeGroup)")
plt.title("Model Accuracy Comparison (Titanic)")
plt.ylabel("Accuracy")
plt.ylim(0,1)
plt.legend()
plt.show()

# Visualization 2: Binary Confusion Matrix
fig, axes = plt.subplots(1, 3, figsize=(15,4))
for ax, (name, res) in zip(axes, results_bin.items()):
    sns.heatmap(res["confusion_matrix"], annot=True, fmt='d', cmap="Blues",
                xticklabels=["Not Survived","Survived"], yticklabels=["Not Survived","Survived"], ax=ax)
    ax.set_title(name + " (Binary)")
plt.tight_layout()
plt.show()

# Visualization 3: Multiclass Confusion Matrix
fig, axes = plt.subplots(1, 3, figsize=(15,4))
for ax, (name, res) in zip(axes, results_multi.items()):
    sns.heatmap(res["confusion_matrix"], annot=True, fmt='d', cmap="Greens",
                xticklabels=model.classes_, yticklabels=model.classes_, ax=ax)
    ax.set_title(name + " (Multiclass)")
plt.tight_layout()
plt.show()

# Binary ROC Curves
plt.figure(figsize=(7,6))
for name, res in results_bin.items():
    plt.plot(res["fpr"], res["tpr"], label=f"{name} (AUC={res['auc']:.2f})")
plt.plot([0,1],[0,1],'k--')
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curves - Binary Classification (Survived)")
plt.legend()
plt.show()

NameError: name 'roc_curve' is not defined